In [ ]:
from pyspark.sql import functions as f
from pyspark.sql.functions import *
from datetime import datetime

def join_recebimento(bi_recebimento, bi_acordos_promessas):
    bi_acordos_promessas = bi_acordos_promessas.filter(f.col("NUM_QTDE_PARCELAS") == 1 )
    
    df_bi_recebimento = (bi_recebimento.join(bi_acordos_promessas, 
                           bi_recebimento['cod_parcelamento'] == bi_acordos_promessas['cod_parcelamento'], "left"))
    return df_bi_recebimento.drop(bi_acordos_promessas['cod_parcelamento']).drop(bi_acordos_promessas['COD_CONTRATO'])
  
  
def join_carteira_cobranca(df_bi_recebimento, bi_carteira_cobranca):
    df_bi_recebimento = (df_bi_recebimento.join(bi_carteira_cobranca, 
                           df_bi_recebimento['COD_CONTRATO'] == bi_carteira_cobranca['COD_CONTRATO'], "left"))
    return df_bi_recebimento.drop(bi_carteira_cobranca['COD_CONTRATO']).drop(bi_carteira_cobranca['TIP_MATRICULA'])
  
  
def join_base_zeus(df_bi_recebimento, base_zeus):
    df_bi_recebimento =  (df_bi_recebimento.withColumn("COD_CPFCGC_PES", f.when(f.col("TIP_MATRICULA") == "PRESENCIAL",
                        f.concat(
                        f.lpad(f.col("COD_FILIAL"),3,'0'), f.lit(""), f.substring(f.col('COD_CONTRATO_ORIGEM'), 2, 15)))
                         .otherwise(f.concat(f.lit("EAD"),f.lit(""), f.col("COD_CONTRATO_ORIGEM")))))
    
    df_bi_recebimento = (df_bi_recebimento.join(base_zeus, 
                           df_bi_recebimento['COD_CPFCGC_PES'] == base_zeus['COD_RA_ZEUS'], "left"))
    return df_bi_recebimento.drop(base_zeus['COD_RA_ZEUS'])  


def join_titulos_vertia(df_bi_recebimento, bi_titulos_vertia):
    df_bi_recebimento = (df_bi_recebimento.join(bi_titulos_vertia,
                           f.ltrim(f.rtrim(df_bi_recebimento['COD_CPFCGC_PES'])) == f.ltrim(f.rtrim(bi_titulos_vertia['COD_CPFCGC_PES'])), "left"))
    return df_bi_recebimento.drop(bi_titulos_vertia['COD_CPFCGC_PES']).drop(bi_titulos_vertia["COD_CONTRATO"])  
  
 
def join_originais(df_bi_recebimento, bi_originais):    
    df_bi_recebimento = (df_bi_recebimento.join(bi_originais, 
                           (df_bi_recebimento['COD_PARCELAMENTO'] == bi_originais['COD_PARCELAMENTO_ORI']) &
                           (bi_originais['NOM_NATUREZA_TITULO'] == "MENSALIDADE") &
                           (bi_originais['IND_NATUREZA_REMATRICULA'] == "S"), "left"))
    return df_bi_recebimento.drop(bi_originais['COD_PARCELAMENTO_ORI'])
  

def join_bi_contrato(df_bi_recebimento, bi_contrato):
    df_bi_recebimento = (df_bi_recebimento.join(bi_contrato, 
                           df_bi_recebimento['COD_CONTRATO'] == bi_contrato['COD_CONTRATO'], "left"))
    return df_bi_recebimento.drop(bi_contrato['COD_CONTRATO'])   
    
    
def join_polos_unidades(df_bi_recebimento, polos_unidades, bi_carteira_cobranca):
    df_bi_recebimento = df_bi_recebimento.drop(bi_carteira_cobranca["COD_FILIAL"])
    df_bi_recebimento = (df_bi_recebimento.join(polos_unidades, 
                           (df_bi_recebimento['COD_FILIAL'] == polos_unidades['COD_CLBR']) &
                           (df_bi_recebimento['COD_FILIAL'] == polos_unidades['COD_OLMP']), "left"))
    
    return df_bi_recebimento.drop(polos_unidades['COD_OLMP']).drop(polos_unidades['COD_CLBR'])
  
      
def execute(spark, inputs, parameters=None, dynamic_parameters=None, write=True):
    polos_unidades = inputs['df_polos_parceiros_e_proprios']
    bi_acordos_promessas = inputs['df_bi_acordos_promessas']
    bi_originais = inputs['df_bi_originais']
    bi_titulos_vertia = inputs['df_titulos_vertia']
    base_zeus = inputs['df_base_zeus']
    bi_recebimento = inputs['df_bi_recebimentos']
    bi_contrato = inputs['df_bi_contrato']
    bi_carteira_cobranca = inputs['df_bi_carteira_cobranca']
    
    
    df_bi_recebimento = join_recebimento(bi_recebimento, bi_acordos_promessas)
    df_bi_recebimento = join_carteira_cobranca(df_bi_recebimento, bi_carteira_cobranca)
    df_bi_recebimento = join_base_zeus(df_bi_recebimento, base_zeus)
    df_bi_recebimento = join_titulos_vertia(df_bi_recebimento, bi_titulos_vertia)
    df_bi_recebimento = join_originais(df_bi_recebimento, bi_originais)
    df_bi_recebimento = join_bi_contrato(df_bi_recebimento, bi_contrato)
    df_bi_recebimento = join_polos_unidades(df_bi_recebimento, polos_unidades, bi_carteira_cobranca)


    df_bi_recebimento = (df_bi_recebimento
    .drop(bi_contrato["NOM_MARCA_CONTRATO"])
    .drop(bi_originais["STA_TITULO"])
    .drop(bi_titulos_vertia["STA_TITULO"])
    .drop(bi_contrato["TIP_MATRICULA"])
    .drop(bi_titulos_vertia["DAT_PAGAMENTO"])
    )
    
    df_bi_recebimento = (df_bi_recebimento.filter(f.col("STA_TITULO") == "PAGO"))    


    df_bi_recebimento = (df_bi_recebimento
                            .withColumn("DSC_CANAL_REPORT",
                            f.when(f.ltrim(f.rtrim(f.col("NOM_USUARIO_ACORDO"))) == "raid-portal-aluno", "raid-portal-aluno")
                            .when((f.col("NOM_GRUPO_USUARIO").isNull()) & (f.ltrim(f.rtrim(f.col("NOM_USUARIO_ACORDO"))) == "adm"), "adm")
                            .otherwise(
                                f.col("NOM_USUARIO_ACORDO")
                        )))

    df_bi_recebimento = (df_bi_recebimento
                            .withColumn("NOM_CARTEIRA",
                            f.when(("TIP_MATRICULA" == f.lit("TIP_MATRICULA")) & (f.col("NOM_MARCA_CONTRATO").isin('ANHANGUERA','UNIDERP')), "AEDU PRESENCIAL")
                            .when(("TIP_MATRICULA" == f.lit("EAD")) & (f.col("NOM_MARCA_CONTRATO").isin('ANHANGUERA','UNIDERP')), "AEDU EAD")
                            .when(("TIP_MATRICULA" == f.lit("PRESENCIAL")) & (f.col("NOM_MARCA_CONTRATO").isin('FAMA','PITAGORAS','UNIC','UNIME','UNOPAR','UNIRONDON')), "DM PRESENCIAL")
                            .when(("TIP_MATRICULA" == f.lit("EAD")) & (f.col("TIP_MATRICULA").isin('UNIME','UNOPAR','PITAGORAS','UNIASSELVI','PITAGORAS','FAMA-MACAPA','UNIC','UNIRONDON')), "UNOPAR EAD")
                        ))

    df_bi_recebimento = (df_bi_recebimento.withColumn("NOM_CANAL_COMISSAO", f.lit("RAID_ATP_INTERVALOR")))

    df_bi_recebimento = (df_bi_recebimento.filter(
                        (f.lower(f.col("NOM_USUARIO_ACORDO")).like("%-mal%")) &
                        (f.lower(f.col("NOM_GRUPO_USUARIO")).like("%atp_malta%")) &
                        (f.col("DAT_PAGAMENTO") >= '2021-09-10')
                        ))

    df_bi_recebimento = (df_bi_recebimento.withColumn("CARTEIRA", f.lit("VERTIA")))

    df_bi_recebimento = (df_bi_recebimento.withColumn("NATUREZA_TITULO", f.lit("MENSALIDADE")))
    df_bi_recebimento = (df_bi_recebimento.withColumn("NATUREZA_REMATRICULA", f.lit("S")))

    df_bi_recebimento = df_bi_recebimento.filter(
                                                (f.col("NUM_DIAS_ATRASO").isNull())|
                                                (f.col("NUM_DIAS_ATRASO") == 0)
                                                )

    df_bi_recebimento = (df_bi_recebimento.select(
        df_bi_recebimento["NOM_CARTEIRA"],
        df_bi_recebimento["DSC_CANAL_REPORT"],
        bi_acordos_promessas["STA_PARCELAMENTO"],
        bi_acordos_promessas["VLR_PRINCIPAL"],
        bi_acordos_promessas["VLR_DESC_PRINCIPAL"],
        bi_acordos_promessas["VLR_MULTA"],
        bi_acordos_promessas["VLR_DESC_MULTA"],
        bi_acordos_promessas["VLR_JUROS_MORA"],
        bi_acordos_promessas["VLR_JUROS_FINANC"],
        bi_acordos_promessas["VLR_DESC_JUROS_FINANC"],
        bi_acordos_promessas["VLR_CORRECAO_MONETARIA"],
        bi_acordos_promessas["VLR_DESC_CORRECAO_MONETARIA"],
        base_zeus["TIP_CURSO_ZEUS"],
        bi_acordos_promessas["VLR_DESC_TAXA_NEGOCIACAO"],
        bi_acordos_promessas["VLR_TAXA_NEGOCIACAO"],
        bi_acordos_promessas["NUM_PAGAMENTO_ENTRADA"],
        bi_recebimento["VLR_PARCELA"],
        df_bi_recebimento["NUM_DIAS_ATRASO"],
        bi_acordos_promessas["COD_CLIENTE"],
        bi_acordos_promessas["COD_CONTATO"],
        df_bi_recebimento["COD_PARCELAMENTO"],
        df_bi_recebimento["TIP_MATRICULA"],
        bi_recebimento["NOM_PRODUTO_PEP"],
        bi_originais["COD_TITULO"],
        bi_recebimento["NOM_GRUPO_USUARIO"],
        bi_recebimento["NOM_USUARIO_ACORDO"],
        bi_originais["DAT_VENCIMENTO"],
        bi_originais["NUM_PARCELA"],
        bi_recebimento["DAT_PAGAMENTO"],
        bi_recebimento["VLR_PAGAMENTO"],
        bi_acordos_promessas["DSC_NATUREZA_REMATRICULA"],
        bi_recebimento["TIP_NATUREZA_TITULO"],
        bi_recebimento["DSC_NATUREZA_TITULO"],
        bi_originais["NUM_MES_COMPETENCIA"],
        bi_originais["NUM_ANO_COMPETENCIA"]
        ))


    return df_bi_recebimento

In [1]:
from pyspark.sql import functions as f
from pyspark.sql.functions import *
from pyspark.sql import Window
import logging

In [ ]:
bi_contrato = spark.read.parquet ('s3a://cdl-planejamentocaixa-us-east-1-660894085742-dev-integration/BI_CONTRATO/')

base_zeus = spark.read.parquet ('s3a://cdl-planejamentocaixa-us-east-1-660894085742-dev-integration/BASE_ZEUS/')

titulos_vertia = spark.read.parquet ('s3a://cdl-planejamentocaixa-us-east-1-660894085742-dev-integration/BI_TITULOS_VERTIA/')

bi_originais = spark.read.parquet ('s3a://cdl-planejamentocaixa-us-east-1-660894085742-dev-integration/BI_ORIGINAIS/')

bi_acordos_promessas = spark.read.parquet ('s3a://cdl-planejamentocaixa-us-east-1-660894085742-dev-integration/BI_ACORDOS_PROMESSAS/')

bi_parcelas = spark.read.parquet ('s3a://cdl-planejamentocaixa-us-east-1-660894085742-dev-integration/BI_PARCELAS/')

polos_unidades = spark.read.parquet('s3a://cdl-planejamentocaixa-us-east-1-660894085742-dev-integration/POLOS_PARCEIROS_E_PROPRIOS/')


In [2]:
spark.conf.set("spark.sql.legacy.parquet.datetimeRebaseModeInRead", "CORRECTED")

In [3]:
#LANDING
# ACORDOS
bi_acordos_promessas = spark.read.format("csv").option("delimiter", ";").option("header","true").load('s3a://cdl-planejamentocaixa-us-east-1-660894085742-dev-integration/EXT_ACORDOS/')

11574 [Thread-5] WARN  org.apache.hadoop.metrics2.impl.MetricsConfig  - Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


In [4]:
#LANDING
# PARCELAS
bi_parcelas = spark.read.format("csv").option("delimiter", ";").option("header","true").load('s3a://cdl-planejamentocaixa-us-east-1-660894085742-dev-integration/EXT_PARCELAS/')

In [5]:
#LANDING
# ORIGINAIS
bi_originais = spark.read.format("csv").option("delimiter", ";").option("header","true").load('s3a://cdl-planejamentocaixa-us-east-1-660894085742-dev-integration/EXT_ORIGINAIS/')

In [6]:
df_bi_acordos_promessas = bi_acordos_promessas
df_bi_parcelas = bi_parcelas
df_bi_originais = bi_originais

In [180]:
df_bi_acordos_promessas = (df_bi_acordos_promessas.select(
        f.col("ID_REGISTRO").cast("string").alias("COD_REGISTRO"),
        f.col("TP_MATRICULA").cast("string").alias("TIP_TP_MATRICULA"),
        f.col("MARCA_CONTRATO").cast("string").alias("DSC_MARCA_CONTRATO"),
        f.col("ID_CLIENTE").cast("string").alias("COD_CLIENTE"),
        f.col("NM_CLIENTE").cast("string").alias("NOM_NM_CLIENTE"),
        f.col("ID_CONTATO").cast("string").alias("COD_CONTATO"),
        f.col("CPF_CNPJ").cast("string").alias("DSC_CPF_CNPJ"),
        f.col("ID_PARCELAMENTO").cast("string").alias("COD_PARCELAMENTO"),
        f.col("TP_PARCELAMENTO").cast("string").alias("TIP_TP_PARCELAMENTO"),
        f.col("QTDE_PARCELAS").cast("string").alias("NUM_QTDE_PARCELAS"),
        f.col("VL_TOTAL").cast("string").alias("VLR_TOTAL"),
        f.col("VL_DIVIDA").cast("string").alias("VLR_DIVIDA"),
        f.col("SCORE_GLOBAL").cast("string").alias("DSC_SCORE_GLOBAL"),
        f.col("SCORE_ENGAJAMENTO").cast("string").alias("DSC_SCORE_ENGAJAMENTO"),
        f.col("DESC_CLUSTER").cast("string").alias("DSC_DESC_CLUSTER"),
        f.col("SITUACAO_CONTRATO").cast("string").alias("DSC_SITUACAO_CONTRATO"),
        f.col("VL_PRINCIPAL").cast("string").alias("VLR_PRINCIPAL"),
        f.col("VL_DESC_PRINCIPAL").cast("string").alias("VLR_DESC_PRINCIPAL"),
        f.col("VL_MULTA").cast("string").alias("VLR_MULTA"),
        f.col("VL_DESC_MULTA").cast("string").alias("VLR_DESC_MULTA"),
        f.col("VL_JUROS_MORA").cast("string").alias("VLR_JUROS_MORA"),
        f.col("VL_DESC_JUROS_MORA").cast("string").alias("VLR_DESC_JUROS_MORA"),
        f.col("VL_JUROS_FINANC").cast("string").alias("VLR_JUROS_FINANC"),
        f.col("VL_DESC_JUROS_FINANC").cast("string").alias("VLR_DESC_JUROS_FINANC"),
        f.col("VL_CORRECAO_MONETARIA").cast("string").alias("VLR_CORRECAO_MONETARIA"),
        f.col("VL_DESC_CORRECAO_MONETARIA").cast("string").alias("VLR_DESC_CORRECAO_MONETARIA"),
        f.col("AGENCIA_EXTERNA").cast("string").alias("DSC_AGENCIA_EXTERNA"),
        f.col("GRUPO_USUARIO").cast("string").alias("DSC_GRUPO_USUARIO"),
        f.col("DESC_ATENDIMENTO").cast("string").alias("DSC_ATENDIMENTO"),
        f.col("ID_CONTRATO").cast("string").alias("COD_CONTRATO"),
        f.col("DT_CANCELAMENTO").cast("string").alias("DAT_CANCELAMENTO"),
        f.col("ST_PARCELAMENTO").cast("string").alias("STA_PARCELAMENTO"),
        f.col("US_CRIACAO_PARCELAMENTO").cast("string").alias("NOM_US_CRIACAO_PARCELAMENTO"),
        f.col("DT_CRIACAO_PARCELAMENTO").cast("string").alias("DAT_CRIACAO_PARCELAMENTO"),
        f.col("ID_PAGAMENTO_ENTRADA").cast("string").alias("NUM_PAGAMENTO_ENTRADA"),
        f.col("DT_CRIACAO").cast("string").alias("DAT_CRIACAO"),
        f.col("US_CRIACAO").cast("string").alias("NOM_US_CRIACAO"),
        f.col("DT_MODIFICACAO").cast("string").alias("DAT_MODIFICACAO"),
        f.col("US_MODIFICACAO").cast("string").alias("DSC_US_MODIFICACAO"),
        f.col("NATUREZA_REMATRICULA").cast("string").alias("DSC_NATUREZA_REMATRICULA"),
        f.col("FL_STATUS").cast("string").alias("IND_FL_STATUS"),
        f.to_date(f.col("DT_PROCESSAMENTO"), "dd/MM/yyyy").alias("DAT_PROCESSAMENTO"),
        f.col("VL_TAXA_NEGOCIACAO").cast("string").alias("VLR_TAXA_NEGOCIACAO"),	
        f.col("VL_DESC_TAXA_NEGOCIACAO").cast("string").alias("VLR_DESC_TAXA_NEGOCIACAO"),	
        f.col("ST_CANCELAMENTO").cast("string").alias("STA_CANCELAMENTO")
    ))

In [181]:
win = Window.partitionBy(f.col("COD_PARCELAMENTO")).orderBy(f.col("DAT_PROCESSAMENTO").desc())  
   
df_bi_acordos_promessas = (df_bi_acordos_promessas
                                    .withColumn("row", f.row_number()
                                    .over(win))
                                    .filter((f.col("row") == '1')))  
	 
df_bi_acordos_promessas = df_bi_acordos_promessas.select(f.col("*"), f.expr("CASE WHEN STA_PARCELAMENTO = 'Payment' THEN 'EM ABERTO' " + 
               "WHEN STA_PARCELAMENTO = 'Concluded' THEN 'PAGO'" +
			   "WHEN STA_PARCELAMENTO = 'Approved' THEN 'PROMESSA'" +
			   "WHEN STA_PARCELAMENTO = 'Cancelled' THEN 'CANCELADO' WHEN STA_PARCELAMENTO IS NULL THEN ''" +
               "ELSE STA_PARCELAMENTO END").alias("STA_PARCELAMENTO_NEW"))
  
df_bi_acordos_promessas = df_bi_acordos_promessas.select(f.col("*"), f.expr("CASE WHEN DSC_ATENDIMENTO = 'raid-portal-aluno' THEN 'WEB' " + 
               "WHEN DSC_ATENDIMENTO = 'adm' THEN 'ATP Próprio/Demais canais'" +
               "ELSE DSC_ATENDIMENTO END").alias("NOM_USUARIO_ACORDO"))  
	
	
#layout
df_bi_acordos_promessas = df_bi_acordos_promessas.select('DAT_PROCESSAMENTO','COD_REGISTRO','TIP_TP_MATRICULA','DSC_MARCA_CONTRATO','COD_CLIENTE','NOM_NM_CLIENTE','COD_CONTATO','DSC_CPF_CNPJ','COD_PARCELAMENTO','TIP_TP_PARCELAMENTO','NUM_QTDE_PARCELAS','VLR_TOTAL','VLR_DIVIDA','DSC_SCORE_GLOBAL','DSC_SCORE_ENGAJAMENTO','DSC_DESC_CLUSTER','DSC_SITUACAO_CONTRATO','VLR_PRINCIPAL','VLR_DESC_PRINCIPAL','VLR_MULTA','VLR_DESC_MULTA','VLR_JUROS_MORA','VLR_DESC_JUROS_MORA','VLR_JUROS_FINANC','VLR_DESC_JUROS_FINANC','VLR_CORRECAO_MONETARIA','VLR_DESC_CORRECAO_MONETARIA','DSC_AGENCIA_EXTERNA','DSC_GRUPO_USUARIO','DSC_ATENDIMENTO','COD_CONTRATO','DAT_CANCELAMENTO','STA_PARCELAMENTO','NOM_US_CRIACAO_PARCELAMENTO','DAT_CRIACAO_PARCELAMENTO','NUM_PAGAMENTO_ENTRADA','DAT_CRIACAO','NOM_US_CRIACAO','DAT_MODIFICACAO','DSC_US_MODIFICACAO','DSC_NATUREZA_REMATRICULA','IND_FL_STATUS','VLR_TAXA_NEGOCIACAO','VLR_DESC_TAXA_NEGOCIACAO','STA_CANCELAMENTO')

In [182]:
df_bi_parcelas = (df_bi_parcelas.select(
        f.col("ID_REGISTRO").cast("string").alias("COD_REGISTRO"),
		f.col("ID_TITULO").cast("string").alias("COD_TITULO"),
		f.col("NO_TITULO").cast("string").alias("NUM_TITULO"),
		f.col("NATUREZA_TITULO").cast("string").alias("NATUREZA_TITULO"),
		f.col("DESC_NATUREZA_TITULO").cast("string").alias("DESC_NATUREZA_TITULO"),
		f.col("DT_PAGAMENTO").cast("string").alias("DAT_PAGAMENTO"),
		f.col("VL_PAGAMENTO").cast("string").alias("VLR_PAGAMENTO"),
		f.col("ST_TITULO").cast("string").alias("STA_TITULO"),
		f.col("ID_CLIENTE").cast("string").alias("COD_CLIENTE"),
		f.col("ID_CONTRATO").cast("string").alias("COD_CONTRATO"),
		f.col("ID_PARCELAMENTO").cast("string").alias("COD_PARCELAMENTO_PAR"),
		f.col("ID_PARCELA").cast("string").alias("COD_PARCELA"),
		f.col("ID_PAGAMENTO_ENTRADA").cast("string").alias("COD_PAGAMENTO_ENTRADA"),
		f.col("DT_VENCIMENTO").cast("string").alias("DAT_VENCIMENTO"),
		f.col("VL_PARCELA").cast("string").alias("VLR_PARCELA"),
		f.col("VL_PRINCIPAL").cast("string").alias("VLR_PRINCIPAL"),
		f.col("VL_DESC_PRINCIPAL").cast("string").alias("VLR_DESC_PRINCIPAL"),
		f.col("VL_MULTA").cast("string").alias("VLR_MULTA"),
		f.col("VL_DESC_MULTA").cast("string").alias("VLR_DESC_MULTA"),
		f.col("VL_JUROS_MORA").cast("string").alias("VLR_JUROS_MORA"),
		f.col("VL_DESC_JUROS_MORA").cast("string").alias("VLR_DESC_JUROS_MORA"),
		f.col("VL_JUROS_FINANC").cast("string").alias("VLR_JUROS_FINANC"),
		f.col("VL_DESC_JUROS_FINANC").cast("string").alias("VLR_DESC_JUROS_FINANC"),
		f.col("VL_CORRECAO_MONETARIA").cast("string").alias("VLR_CORRECAO_MONETARIA"),
		f.col("VL_DESC_CORRECAO_MONETARIA").cast("string").alias("VLR_DESC_CORRECAO_MONETARIA"),
		f.col("ST_PROCESSAMENTO").cast("string").alias("ST_PROCESSAMENTO"),
		f.col("DT_CRIACAO").cast("string").alias("DAT_CRIACAO"),
		f.col("US_CRIACAO").cast("string").alias("NOM_US_CRIACAO"),
		f.col("DT_MODIFICACAO").cast("string").alias("DAT_MODIFICACAO"),
		f.col("US_MODIFICACAO").cast("string").alias("NOM_US_MODIFICACAO"),
		f.col("FL_STATUS").cast("string").alias("IND_STATUS"),
		f.to_date(f.col("DT_PROCESSAMENTO"), "dd/MM/yyyy").alias("DAT_PROCESSAMENTO")
    ))

In [183]:
#group parcelas
win = Window.partitionBy(f.col("COD_PARCELAMENTO_PAR"),f.col("COD_PARCELA")).orderBy(f.col("DAT_PROCESSAMENTO").desc())  
 
df_bi_parcelas = (df_bi_parcelas
                   .withColumn("row", f.row_number().over(win))
                   .filter((f.col("row") == '1')))  
                       
df_bi_parcelas = df_bi_parcelas.select('COD_CONTRATO','COD_CLIENTE','COD_PARCELAMENTO_PAR','COD_PARCELA','DAT_PROCESSAMENTO','COD_TITULO','NUM_TITULO','DAT_PAGAMENTO','VLR_PAGAMENTO','DAT_VENCIMENTO','VLR_PARCELA')
    

In [7]:
df_bi_originais = (df_bi_originais.select(
        f.col("ID_REGISTRO").cast("string ").alias("COD_REGISTRO"),
        f.col("ID_TITULO").cast("string").alias("COD_TITULO"),
        f.col("ID_CONTRATO").cast("string").alias("COD_CONTRATO_ORI"),
        f.col("ID_CLIENTE").cast("string").alias("COD_CLIENTE_ORI"),
        f.col("NO_TITULO").cast("string").alias("NUM_TITULO"),
        f.col("NATUREZA_TITULO").cast("string").alias("NOM_NATUREZA_TITULO"),
        f.col("DESC_NATUREZA_TITULO").cast("string").alias("DSC_NATUREZA_TITULO"),
        f.col("MES_COMPETENCIA").cast("string").alias("NUM_MES_COMPETENCIA"),
        f.col("ANO_COMPETENCIA").cast("string").alias("NUM_ANO_COMPETENCIA"),
        f.col("VL_LIQUIDO").cast("string").alias("VLR_LIQUIDO"),
        f.col("VL_SEM_DESC_PONTUAL").cast("string").alias("VLR_SEM_DESC_PONTUAL"),
        f.col("DT_VENCIMENTO").cast("string").alias("DAT_VENCIMENTO"),
        f.col("NO_PARCELA").cast("string").alias("NUM_PARCELA"),
        f.col("ID_PARCELAMENTO").cast("string").alias("COD_PARCELAMENTO_ORI"),
        f.col("DT_EMISSAO").cast("string").alias("DAT_EMISSAO"),
        f.col("NATUREZA_REMATRICULA").cast("string").alias("IND_NATUREZA_REMATRICULA"),
        f.col("DT_PAGAMENTO").cast("string").alias("DAT_PAGAMENTO"),
        f.col("VL_PAGAMENTO").cast("string").alias("VLR_PAGAMENTO"),
        f.col("ST_TITULO").cast("string").alias("STA_TITULO"),
        f.col("DT_CADASTRO").cast("string").alias("DAT_CADASTRO"),
        f.col("ST_PROCESSAMENTO").cast("string").alias("STA_PROCESSAMENTO"),
        f.col("DT_CRIACAO").cast("string").alias("DAT_CRIACAO"),
        f.col("US_CRIACAO").cast("string").alias("NOM_US_CRIACAO"),
        f.col("DT_MODIFICACAO").cast("string").alias("DAT_MODIFICACAO"),
        f.col("US_MODIFICACAO").cast("string").alias("NOM_US_MODIFICACAO"),
        f.col("NO_MANUTENCAO_NEGATIVACAO").cast("string").alias("NUM_MANUTENCAO_NEGATIVACAO"),
        f.col("FL_STATUS").cast("string").alias("IND_STATUS"),
        f.to_date(f.col("DT_PROCESSAMENTO"), "dd/MM/yyyy").alias("DAT_PROCESSAMENTO_ORI"),
        f.col("NM_GRUPO_TELA_REMAT").cast("string").alias("NOM_GRUPO_TELA_REMAT"),
        f.col("NR_CONTADOR_TELA_REMAT").cast("string").alias("NUM_CONTADOR_TELA_REMAT")
    ))


ANTLR Tool version 4.8 used for code generation does not match the current runtime version 4.7ANTLR Runtime version 4.8 used for parser compilation does not match the current runtime version 4.7ANTLR Tool version 4.8 used for code generation does not match the current runtime version 4.7ANTLR Runtime version 4.8 used for parser compilation does not match the current runtime version 4.7

In [8]:
win = Window.partitionBy(f.col("COD_TITULO")).orderBy(f.col("DAT_PROCESSAMENTO_ORI").desc())  
df_bi_originais = (df_bi_originais
                            .withColumn("row", f.row_number().over(win))
                            .filter((f.col("row") == '1')))  
    
    
df_bi_originais = df_bi_originais.select('COD_TITULO','COD_CLIENTE_ORI','COD_CONTRATO_ORI','COD_PARCELAMENTO_ORI','NUM_MES_COMPETENCIA','NUM_ANO_COMPETENCIA','NUM_PARCELA','IND_NATUREZA_REMATRICULA','STA_TITULO','DAT_PROCESSAMENTO_ORI','NOM_GRUPO_TELA_REMAT','NUM_CONTADOR_TELA_REMAT','NOM_NATUREZA_TITULO','DSC_NATUREZA_TITULO','VLR_LIQUIDO','DAT_VENCIMENTO')

In [9]:
bi_acordos_promessas = df_bi_acordos_promessas
bi_parcelas = df_bi_parcelas
bi_originais = df_bi_originais

In [187]:
# ORIGINAIS GROUP 
def join_originais_group(bi_originais):
    win = Window.partitionBy(f.col("COD_TITULO")).orderBy(f.col("DAT_PROCESSAMENTO_ORI").desc())  
    df_bi_originais = (bi_originais
                            .withColumn("row", f.row_number().over(win))
                            .filter((f.col("row") == '1'))) 
    
    df_bi_originais = (df_bi_originais.select(f.col('COD_TITULO'),
                                              f.col('COD_CONTRATO_ORI'),
                                              f.col('COD_PARCELAMENTO_ORI'),
                                              f.col('NUM_MES_COMPETENCIA'),
                                              f.col('NUM_ANO_COMPETENCIA'),
                                              f.col('NUM_PARCELA'),
                                              f.col('IND_NATUREZA_REMATRICULA'),
                                              f.col('STA_TITULO'),
                                              f.col('DSC_NATUREZA_TITULO'),
                                              f.col('NOM_NATUREZA_TITULO'),
                                              f.col('DAT_VENCIMENTO')
                                             ))

    
    return df_bi_originais

In [12]:
df_bi_originais.select(f.to_date(date_format(f.col('DAT_VENCIMENTO'),"yyyy-MM-dd"),"yyyy-MM-dd")).show()

+------------------------------------------------------------+
|to_date(date_format(DAT_VENCIMENTO, yyyy-MM-dd), yyyy-MM-dd)|
+------------------------------------------------------------+
|                                                  2021-10-27|
|                                                  2021-08-18|
|                                                  2022-01-27|
|                                                  2022-01-19|
|                                                  2021-08-23|
|                                                  2022-01-20|
|                                                  2021-11-24|
|                                                  2021-11-24|
|                                                  2021-10-24|
|                                                  2021-08-24|
|                                                  2022-03-24|
|                                                  2021-10-24|
|                                                  2021

In [188]:
# ACORDOS GROUP 
def join_acordos_group(bi_acordos_promessas):    
    win = Window.partitionBy(f.col("COD_PARCELAMENTO")).orderBy(f.col("DAT_PROCESSAMENTO").desc())   
    df_bi_acordos_promessas = (bi_acordos_promessas
                                    .withColumn("row", f.row_number()
                                    .over(win))
                                    .filter((f.col("row") == '1')))  
    
    df_bi_acordos_promessas = df_bi_acordos_promessas.select(f.col("DAT_PROCESSAMENTO"),
                                                             f.col("TIP_TP_MATRICULA"),
                                                             f.col("COD_CONTRATO"),
                                                             f.col("DSC_MARCA_CONTRATO"),
                                                             f.col("COD_CLIENTE"),
                                                             f.col("NOM_NM_CLIENTE"),
                                                             f.col("COD_PARCELAMENTO"),
                                                             f.col("DAT_CRIACAO_PARCELAMENTO"),
                                                             f.col("TIP_TP_PARCELAMENTO"),
                                                             f.col("NUM_QTDE_PARCELAS"),
                                                             f.col("VLR_TOTAL"),
                                                             f.col("VLR_PRINCIPAL"),
                                                             f.col("VLR_DESC_PRINCIPAL"),
                                                             f.col("VLR_MULTA"),
                                                             f.col("VLR_DESC_MULTA"),
                                                             f.col("VLR_JUROS_MORA"),
                                                             f.col("VLR_JUROS_FINANC"),
                                                             f.col("VLR_DESC_JUROS_FINANC"),
                                                             f.col("VLR_CORRECAO_MONETARIA"),
                                                             f.col("VLR_DESC_CORRECAO_MONETARIA"),
                                                             f.col("VLR_TAXA_NEGOCIACAO"),
                                                             f.col("VLR_DESC_TAXA_NEGOCIACAO"),
                                                             f.col("DSC_GRUPO_USUARIO"),
                                                             f.col("DSC_ATENDIMENTO"),
                                                             f.col("NOM_US_CRIACAO_PARCELAMENTO"),
                                                             f.col("STA_CANCELAMENTO")
                                                            )

    return df_bi_acordos_promessas

In [189]:
# PARCELAS GROUP 
def join_parcelas_group(bi_parcelas):    
    win = Window.partitionBy(f.col("COD_PARCELAMENTO_PAR"),f.col("COD_PARCELA")).orderBy(f.col("DAT_PROCESSAMENTO").desc())   
    df_bi_parcelas = (bi_parcelas
                   .withColumn("row", f.row_number().over(win))
                   .filter((f.col("row") == '1')))   
    
    df_bi_parcelas = df_bi_parcelas.select(f.col('COD_PARCELAMENTO_PAR'),
                                           f.col('COD_PARCELA'),
                                           f.col('COD_CONTRATO'),
                                           f.col('COD_TITULO'),
                                           f.col('NUM_TITULO'),
                                           f.col('DAT_PAGAMENTO'),
                                           f.col('VLR_PAGAMENTO'),
                                           f.col('DAT_VENCIMENTO'),
                                           f.col('VLR_PARCELA')
                                          )
    
    return df_bi_parcelas

In [190]:
df_acordos_promessas = join_acordos_group(bi_acordos_promessas)
df_parcelas = join_parcelas_group(bi_parcelas)
df_originais = join_originais_group(bi_originais)

In [191]:
def join_parcelas(bi_acordos_promessas, bi_parcelas):
    df_acordos_promessas = (bi_acordos_promessas.join(bi_parcelas,
                                  bi_acordos_promessas['COD_PARCELAMENTO'] == bi_parcelas['COD_PARCELAMENTO_PAR'], "left"))
      
    
    return (df_acordos_promessas
            .drop(bi_parcelas['COD_CONTRATO'])
            .drop(bi_acordos_promessas['DAT_PROCESSAMENTO'])
            .drop(bi_parcelas['COD_PARCELAMENTO_PAR']) 
            .drop(bi_acordos_promessas['COD_CLIENTE'])
           )
    

In [192]:
df_acordos_promessas = join_parcelas(bi_acordos_promessas, bi_parcelas)

In [193]:
# TIP PARCELA
df_acordos_promessas = (df_acordos_promessas
                            .withColumn("TIP_PARCELA",
                            f.when((f.col('COD_PARCELA') == "1") & (f.col("VLR_PAGAMENTO").isNull()), "PROMESSA")
                            .when((f.col('COD_PARCELA') == "1") & (~(f.col("VLR_PAGAMENTO").isNull())), "ACORDO")
                            .when((~(f.col('COD_PARCELA') == "1")) & ((f.col("VLR_PAGAMENTO").isNull())) & (f.to_date(date_format(f.col('DAT_VENCIMENTO'),"yyyy-MM-dd"),"yyyy-MM-dd") > f.current_date()), "COLCHAO")
                            .when((~(f.col('COD_PARCELA') == "1")) & ((f.col("VLR_PAGAMENTO").isNull())) & (f.to_date(date_format(f.col('DAT_VENCIMENTO'),"yyyy-MM-dd"),"yyyy-MM-dd") < f.current_date()), "COLCHAO EM ATRASO")
                            .when((~(f.col('COD_PARCELA') == "1")) & (~(f.col("VLR_PAGAMENTO").isNull())) , "COLCHAO PAGO")
                        ))

In [194]:
# NOME SISTEMA
df_acordos_promessas =  (df_acordos_promessas.withColumn("NOM_SISTEMA", f.when(f.substring(f.col("COD_CONTRATO"), 1, 6) == f.lit("OLIMPO"),
                                                     f.lit("OLIMPO"))
                                            .otherwise(
                                                f.concat(f.lit("COLABORAR"))
                                            )))

In [66]:
df_acordos_promessas.select(f.col('NOM_SISTEMA')).distinct().show()

+-----------+
|NOM_SISTEMA|
+-----------+
|  COLABORAR|
|     OLIMPO|
+-----------+



In [195]:
df = df_acordos_promessas

In [ ]:
ciclo_remat = spark.read.parquet('s3a://cdl-planejamentocaixa-us-east-1-660894085742-dev-integration/CICLO_REMAT/')

In [196]:
#-- 2� EMPACOTAMENTO
def join_empac(df_acordos_promessas, bi_originais, ciclo_remat):
    
    df_empac = (bi_originais.filter(
                                (f.col('NOM_NATUREZA_TITULO') == 'MENSALIDADE') &
                                (f.col('IND_NATUREZA_REMATRICULA') == 'S' ) &
                                (f.col('NUM_ANO_COMPETENCIA') == f.year(f.current_date()))
                    ))
    
    df_acordos_promessas = (df_acordos_promessas.join(df_empac,
                                  df_acordos_promessas['COD_PARCELAMENTO'] == df_empac['COD_PARCELAMENTO_ORI'], "left"))

    df_acordos_promessas = (df_acordos_promessas.join(ciclo_remat,
                                  df_acordos_promessas['NOM_SISTEMA'] == ciclo_remat['NOM_SISTEMA'], "left"))
                
                
     
    return (df_acordos_promessas
            .drop(df_empac['DAT_VENCIMENTO'])
            .drop(df_empac['COD_TITULO'])
            .drop(df_empac['COD_CLIENTE_ORI'])
            .drop(df_empac['COD_CONTRATO_ORI'])
            .drop(df_empac['COD_PARCELAMENTO_ORI'])
            .drop(df_empac['STA_TITULO'])
            .drop(df_empac['NUM_PARCELA'])
            #.drop(bi_originais['IND_NATUREZA_REMATRICULA'])
            .drop(df_empac['DAT_PROCESSAMENTO_ORI'])
            .drop(df_empac['NOM_GRUPO_TELA_REMAT'])
            #.drop(bi_originais['NOM_NATUREZA_TITULO'])
            .drop(df_empac['DSC_NATUREZA_TITULO'])
            .drop(df_empac['VLR_LIQUIDO'])
            .drop(df_empac['NUM_CONTADOR_TELA_REMAT'])
            #.drop(ciclo_remat['NOM_CICLO'])
            .drop(ciclo_remat['NOM_SISTEMA'])
            .drop(ciclo_remat['NUM_ANO'])
            .drop(ciclo_remat['NUM_SEMESTRE'])
            #.drop(ciclo_remat['DAT_INICIO'])
            #.drop(ciclo_remat['DAT_FIM'])
           )

In [197]:
df_acordos_promessas = join_empac(df_acordos_promessas, bi_originais, ciclo_remat)

In [204]:
# CODIGO DE EMPAC
df_acordos_promessas = (df_acordos_promessas
                            .withColumn("COD_EMPAC",
                            f.when((f.to_date(date_format(f.col('DAT_CRIACAO_PARCELAMENTO'),"yyyy-MM-dd"),"yyyy-MM-dd") >= f.to_date(date_format(f.col('DAT_INICIO'),"yyyy-MM-dd"),"yyyy-MM-dd")) & (f.to_date(date_format(f.col('DAT_CRIACAO_PARCELAMENTO'),"yyyy-MM-dd"),"yyyy-MM-dd") <= f.to_date(date_format(f.col('DAT_INICIO'),"yyyy-MM-dd"),"yyyy-MM-dd")), f.col('NOM_CICLO'))
                  ))

In [ ]:
df_acordos_promessas.select(f.col('COD_EMPAC')).distinct().show()

In [205]:
df_bi_originais.printSchema()

root
 |-- COD_TITULO: string (nullable = true)
 |-- COD_CLIENTE_ORI: string (nullable = true)
 |-- COD_CONTRATO_ORI: string (nullable = true)
 |-- COD_PARCELAMENTO_ORI: string (nullable = true)
 |-- NUM_MES_COMPETENCIA: string (nullable = true)
 |-- NUM_ANO_COMPETENCIA: string (nullable = true)
 |-- NUM_PARCELA: string (nullable = true)
 |-- IND_NATUREZA_REMATRICULA: string (nullable = true)
 |-- STA_TITULO: string (nullable = true)
 |-- DAT_PROCESSAMENTO_ORI: date (nullable = true)
 |-- NOM_GRUPO_TELA_REMAT: string (nullable = true)
 |-- NUM_CONTADOR_TELA_REMAT: string (nullable = true)
 |-- NOM_NATUREZA_TITULO: string (nullable = true)
 |-- DSC_NATUREZA_TITULO: string (nullable = true)
 |-- VLR_LIQUIDO: string (nullable = true)
 |-- DAT_VENCIMENTO: string (nullable = true)



In [49]:
#-- DIAS ATRASO
def join_empac(df_acordos_promessas, bi_originais):
    
    
    #group by
    df_bi_originais =  (df_bi_originais.groupBy('COD_PARCELAMENTO_ORI')
                                .agg((f.min(f.to_date(f.col('DAT_VENCIMENTO'), 'dd/MM/yyyy'))).alias('DAT_MENOR_VENCIMENTO'))) 
    
    
    
    df_acordos_promessas = (df_acordos_promessas.join(df_empac,
                                  df_acordos_promessas['COD_PARCELAMENTO'] == df_empac['COD_PARCELAMENTO_ORI'], "left"))

    df_acordos_promessas = (df_acordos_promessas.join(ciclo_remat,
                                  df_acordos_promessas['NOM_SISTEMA'] == ciclo_remat['NOM_SISTEMA'], "left"))
                
                
     
    return (df_acordos_promessas
            .drop(df_empac['DAT_VENCIMENTO'])
            .drop(df_empac['COD_TITULO'])
            .drop(df_empac['COD_CLIENTE_ORI'])
            .drop(df_empac['COD_CONTRATO_ORI'])
            .drop(df_empac['COD_PARCELAMENTO_ORI'])
            .drop(df_empac['STA_TITULO'])
            .drop(df_empac['NUM_PARCELA'])
          
           )

In [ ]:
#dias atraso
df_bi_carteira_cobranca = (df_bi_carteira_cobranca.withColumn("NUM_DIAS_ATRASO", 
                      f.floor(
                          f.datediff(f.current_date(), f.to_date(f.to_date("DAT_MENOR_VENCIMENTO")))
                      ))
                )

In [ ]:
	DROP TABLE IF EXISTS #TEMP02;

	 SELECT   T.cd_ID_CLIENTE	
			, T.cd_ID_CONTRATO
			, O.id_parcelamento

			, AP.DT_CRIACAO_PARCELAMENTO
			, DATEDIFF (day, (MIN(O.DT_VENCIMENTO)), AP.DT_CRIACAO_PARCELAMENTO) AS DIAS_ATRASO
	 INTO #TEMP02
	 FROM RAID.TBL_RECEBIMENTOS T  WITH(NOLOCK)
	 INNER JOIN RAID.EXT_ORIGINAIS O WITH(NOLOCK) ON T.CD_ID_PARCELAMENTO = O.id_parcelamento
	 INNER JOIN RAID.TB_ACORDOS_PROMESSAS AP WITH(NOLOCK) ON AP.CD_ID_PARCELAMENTO = O.ID_PARCELAMENTO AND AP.NR_NO_PARCELA = 1
	 GROUP BY  T.CD_ID_CLIENTE	
			, T.CD_ID_CONTRATO
			, O.id_parcelamento
			, AP.DT_CRIACAO_PARCELAMENTO;

In [50]:
ciclo_remat.show()

+---------+-----------+-------+------------+----------+----------+---------------+
|NOM_CICLO|NOM_SISTEMA|NUM_ANO|NUM_SEMESTRE|DAT_INICIO|   DAT_FIM|   execution_ts|
+---------+-----------+-------+------------+----------+----------+---------------+
|     0722|     OLIMPO|   2022|          02|2022-05-16|2022-11-13|20220509T173032|
|     0122|     OLIMPO|   2022|          01|2021-11-14|2022-05-15|20220509T173032|
|   202201|  COLABORAR|   2022|          01|2021-11-14|2022-05-15|20220509T173032|
|   202202|  COLABORAR|   2022|          02|2022-05-16|2022-11-13|20220509T173032|
+---------+-----------+-------+------------+----------+----------+---------------+



In [ ]:
f.to_date(date_format(f.col('DAT_VENCIMENTO'),"yyyy-MM-dd"),"yyyy-MM-dd")

In [ ]:
def join_originais(df_acordos_promessas, bi_originais):
    
    bi_originais = (bi_originais.filter(
                                (f.col('NOM_NATUREZA_TITULO') == 'MENSALIDADE') &
                                (f.col('IND_NATUREZA_REMATRICULA') == 'S' ) &
                                (f.col('NUM_ANO_COMPETENCIA') == f.year(f.current_date()))
                    ))
    
    df_acordos_promessas = (df_acordos_promessas.join(bi_originais,
                                  df_acordos_promessas['COD_PARCELAMENTO'] == bi_originais['COD_PARCELAMENTO_ORI'], "left"))
    
    
    return (df_acordos_promessas
            .drop(bi_originais['DAT_VENCIMENTO'])
            .drop(bi_originais['COD_TITULO'])
           )


In [ ]:
#-- 2� EMPACOTAMENTO


In [ ]:
bi_acordos_promessas.select(date_format((f.to_date(f.col('DAT_CRIACAO_PARCELAMENTO'), 'dd/MM/yyyy')),"yyyy-mm-dd")).show()

In [31]:
df.select(date_format(f.col('DAT_VENCIMENTO'),"yyyy-MM-dd")).show()

+---------------------------------------+
|date_format(DAT_VENCIMENTO, yyyy-MM-dd)|
+---------------------------------------+
|                             2022-06-18|
|                             2022-04-18|
|                             2022-07-18|
|                             2022-03-18|
|                             2022-05-18|
|                             2022-03-28|
|                             2022-05-28|
|                             2022-04-28|
|                             2022-06-28|
|                             2022-07-28|
|                             2022-06-04|
|                             2022-12-04|
|                             2022-09-04|
|                             2023-01-04|
|                             2022-10-04|
|                             2022-08-04|
|                             2022-11-04|
|                             2022-07-04|
|                             2022-05-04|
|                             2021-08-20|
+---------------------------------

In [29]:
df.select(f.col('COD_PARCELA'),f.col('DAT_VENCIMENTO'),f.col('VLR_PAGAMENTO')).show()   

+-----------+--------------------+-------------+
|COD_PARCELA|      DAT_VENCIMENTO|VLR_PAGAMENTO|
+-----------+--------------------+-------------+
|          5|2022-06-18T00:00:...|         null|
|          3|2022-04-18T00:00:...|         null|
|          6|2022-07-18T00:00:...|         null|
|          2|2022-03-18T00:00:...|         null|
|          4|2022-05-18T00:00:...|         null|
|          2|2022-03-28T00:00:...|         null|
|          4|2022-05-28T00:00:...|         null|
|          3|2022-04-28T00:00:...|         null|
|          5|2022-06-28T00:00:...|         null|
|          6|2022-07-28T00:00:...|         null|
|          3|2022-06-04T00:00:...|         null|
|          9|2022-12-04T00:00:...|         null|
|          6|2022-09-04T00:00:...|         null|
|         10|2023-01-04T00:00:...|         null|
|          7|2022-10-04T00:00:...|         null|
|          5|2022-08-04T00:00:...|         null|
|          8|2022-11-04T00:00:...|         null|
|          4|2022-07

In [ ]:
#-- 2� EMPACOTAMENTO
def join_originais(df_acordos_promessas, bi_originais):
    
    bi_originais = (bi_originais.filter(
                                (f.col('NOM_NATUREZA_TITULO') == 'MENSALIDADE') &
                                (f.col('IND_NATUREZA_REMATRICULA') == 'S' ) &
                                (f.col('NUM_ANO_COMPETENCIA') == f.year(f.current_date()))
                    ))
    
    df_acordos_promessas = (df_acordos_promessas.join(bi_originais,
                                  df_acordos_promessas['COD_PARCELAMENTO'] == bi_originais['COD_PARCELAMENTO_ORI'], "left"))
    
    
    return (df_acordos_promessas
            .drop(bi_originais['DAT_VENCIMENTO'])
            .drop(bi_originais['COD_TITULO'])
           )

In [ ]:
(f.to_date(f.col('DAT_VENCIMENTO'), 'dd/MM/yyyy') > f.current_date())


  .when((~(f.col('COD_PARCELA') == "1")) & ((f.col("VLR_PAGAMENTO").isNull())) & (f.to_date(f.col('DAT_VENCIMENTO'), 'dd/MM/yyyy') > f.current_date()), "COLCHAO")
                            .when((~(f.col('COD_PARCELA') == "1")) & ((f.col("VLR_PAGAMENTO").isNull())) & (f.to_date(f.col('DAT_VENCIMENTO'), 'dd/MM/yyyy') < f.current_date()), "COLCHAO EM ATRASO")
                            

In [ ]:
df_acordos_promessas.printSchema()

In [49]:
df_acordos_promessas.count()

5294479

In [ ]:
df_parcelas = join_parcelas_group(bi_parcelas)
df_originais = join_originais_group(bi_originais)
df_acordos_promessas = join_acordos_group(bi_acordos_promessas)


df_acordos_promessas = join_parcelas(bi_acordos_promessas, bi_parcelas)
df_acordos_promessas = join_originais(df_acordos_promessas, bi_originais)
df_acordos_promessas = join_contratos(df_acordos_promessas, bi_contratos)
df_acordos_promessas = join_zeus(df_acordos_promessas, bi_zeus)
df_acordos_promessas = join_vertia(df_acordos_promessas, bi_titulos_vertia)  
  